In [ ]:
!pip install python-telegram-bot

In [ ]:
from google.colab import files

# Загрузи сюда math_misconception_model.pkl (и answers.pkl, если используешь его)
uploaded = files.upload()

In [ ]:
# ⚠️ ПОДСКАЗКА (мелочь): в model1 ты сохранял answers.pkl и грузишь его выше,
# но здесь словарь задаёшь заново вручную. Реши: либо грузить из .pkl, либо оставить так,
# но тогда загрузка answers.pkl лишняя.
# Ещё глянь ключи Wrong_fraction / Wrong_Fraction — почему их два? Это про регистр в метках.

answers = {
    "No_Misconception": "No misconception was detected. The student's explanation appears to be correct.",
    "Incomplete": "The student's explanation is incomplete and is missing an important part of the reasoning.",
    "Wrong_fraction": "The student may have made an error when working with fractions.",
    "Wrong_Fraction": "The student may have made an error when working with fractions.",
    "Wrong_term": "The student may be using a mathematical term incorrectly.",
    "Additive": "The student may be using addition where another mathematical operation is needed.",
    "Subtraction": "The student may be using subtraction incorrectly.",
    "Division": "The student may be using division incorrectly.",
    "Mult": "The student may be using multiplication incorrectly.",
    "Inversion": "The student may have misunderstood an inverse operation or transformation.",
    "Duplication": "The student may be counting or applying the same value more than once.",
    "Positive": "The student may have misunderstood the sign or positive value.",
    "Scale": "The error may be related to scaling or changing the size of a quantity.",
    "Whole_numbers_larger": "The student may think that a larger whole number always represents a larger quantity.",
    "Not_variable": "The student may have misunderstood the role of a variable.",
    "Wrong_Operation": "The student may have chosen the wrong mathematical operation.",
    "WNB": "The student's reasoning may be incorrect or insufficient.",
    "Irrelevant": "The explanation contains information that is not directly relevant to solving the problem.",
    "Unknowable": "The student may think that the answer cannot be determined from the information given.",
    "Adding_across": "The student adds the numerators and denominators across the fractions instead of using a common denominator."
}

print("Answer dictionary loaded!")

In [ ]:
import joblib

model = joblib.load("math_misconception_model.pkl")

print("✅ Model loaded!")

In [ ]:
from telegram import Update
from telegram.ext import Application, CommandHandler, MessageHandler, filters, ContextTypes

# ======================================================================
# 🔴 ЗАДАНИЕ 1 — БЕЗОПАСНОСТЬ (обязательно)
# Токен нельзя держать прямо в коде: этот уже "засветился".
# 1) Отзови старый токен у @BotFather командой /revoke и выпусти новый.
# 2) Подумай, как ввести токен, чтобы он НЕ оставался в файле.
#    Подсказка: посмотри модуль getpass -> getpass("...").
# ======================================================================
TOKEN = "ВСТАВЬ_ЗДЕСЬ_СВОЙ_НОВЫЙ_ТОКЕН"

async def start(update: Update, context: ContextTypes.DEFAULT_TYPE):
    await update.message.reply_text("🤖 Hello! The bot is working!")

async def message(update: Update, context: ContextTypes.DEFAULT_TYPE):
    text = update.message.text

    prediction = model.predict([text])[0]

    # ==================================================================
    # 🔴 ЗАДАНИЕ 2 — ГЛАВНЫЙ БАГ (из-за него бот молчит на сообщения)
    # Слева и справа от "=" стоит ОДНО И ТО ЖЕ имя: answers.
    # Из-за этого Python считает answers локальной переменной на ВСЮ функцию,
    # и в момент .get(...) она ещё не создана -> UnboundLocalError -> бот падает.
    # (Именно поэтому /start работает, а на текст бот не отвечает.)
    # Вопрос: как назвать РЕЗУЛЬТАТ по-другому, чтобы не затирать словарь answers?
    # ==================================================================
    answers = answers.get(
        prediction,
        "No explanation available for this misconception yet."
    )

    # ⚠️ ЗАДАНИЕ 3 — ЧЕСТНОСТЬ ПРЕДСКАЗАНИЙ
    # В model1 модель училась на text = QuestionText + " " + StudentExplanation,
    # а сюда приходит только сообщение пользователя. Модель видит не то, на чём училась.
    # Падать не будет, но ответы будут неточные. Подумай: что подавать боту,
    # чтобы вход совпадал с обучением?

    await update.message.reply_text(
        f"🧠 Prediction: {prediction}\n\n"
        f"💬 Explanation: {answers}"
    )

app = Application.builder().token(TOKEN).build()

app.add_handler(CommandHandler("start", start))
app.add_handler(MessageHandler(filters.TEXT & ~filters.COMMAND, message))

import nest_asyncio
nest_asyncio.apply()

await app.initialize()
await app.start()
await app.updater.start_polling()

# ⚠️ ЗАДАНИЕ 4 — ПРОВЕРКА
# "Бот запущен!" НЕ значит "бот работает".
# После запуска обязательно напиши боту реальное сообщение и проверь ответ.
print("🤖 Бот запущен!")